<a href="https://colab.research.google.com/github/Silas-Kamanda/Projects/blob/main/copy_of_skw_swahili_asr_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + Augmentation + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2
!pip install torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 --index-url https://download.pytorch.org/whl/cu124
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.44.1
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio

print("Installing PEFT, audiomentations, and wandb...")
!pip install peft==0.14.0
!pip install audiomentations==0.36.0
!pip install wandb==0.18.3

try:
    import json
    import re
    import numpy as np
    import pandas as pd
    import torch
    import torchvision
    import torchaudio
    from torchaudio.transforms import SpecAugment
    import transformers
    import datasets
    import accelerate
    import evaluate
    import wandb
    from tqdm.notebook import tqdm
    from sklearn.model_selection import train_test_split
    from datasets import Dataset, DatasetDict
    from transformers import (
        WhisperProcessor,
        WhisperFeatureExtractor,
        WhisperTokenizerFast,
        WhisperForConditionalGeneration,
        Seq2SeqTrainingArguments,
        Seq2SeqTrainer,
        EarlyStoppingCallback
    )
    from peft import (
        prepare_model_for_kbit_training,
        LoraConfig,
        PeftModel,
        PeftConfig,
        get_peft_model
    )
    from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
    from dataclasses import dataclass
    from typing import Any, Dict, List, Union
    import warnings
    import gradio as gr
    import csv
    import soundfile as sf
    import gc
    warnings.filterwarnings("ignore")
except ImportError as e:
    print(f"Import error: {e}")
    raise

print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Torchaudio version: {torchaudio.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"Accelerate version: {accelerate.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"WandB version: {wandb.__version__}")
try:
    import peft
    print(f"PEFT version: {peft.__version__}")
    import audiomentations
    print(f"Audiomentations version: {audiomentations.__version__}")
except ImportError as e:
    print(f"Failed to import library: {e}")
    raise ImportError("Required library missing. Check installation logs.")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Device Count: {torch.cuda.device_count()}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-qkvout-fc1fc2-aug-9k-samples",
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "GaussianNoise+TimeStretch+PitchShift+SpecAugment",
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
print(f"CSV found at: {csv_path}")
print(f"Audio base directory: {audio_base_path}")
print(f"Model outputs to: {output_dir}")
os.makedirs(output_dir, exist_ok=True)

try:
    df = pd.read_csv(csv_path).head(9000)
    print(f"Initial dataframe shape: {df.shape}")
except FileNotFoundError:
    print(f"Error loading CSV from {csv_path}. Upload swahili_dataset.csv to /content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/")
    raise

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r'[^\w\s\']', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]
print(f"Shape after cleaning: {df.shape}")

try:
    print("Skipping audio validation to save time. Assuming all audio files are valid.")
    if len(df) == 0: raise ValueError("No valid samples after transcript cleaning.")
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
    del df, temp_df; gc.collect()
except Exception as e_split:
    print(f"Dataset splitting error: {e_split}")
    raise

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"\nDataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")
if len(raw_datasets["train"]) > 0: print(f"Sample 0 (raw): {raw_datasets['train'][0]}")
print(f"Raw dataset columns: {raw_datasets['train'].features}")
print("-" * 30)

model_name_loaded = "openai/whisper-small"
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
try:
    feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
    tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
    processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
    print(f"Whisper Processor for {language} ({language_abbr}), task: {task} loaded from {model_name_loaded}.")
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id
        print(f"Set processor.tokenizer.pad_token_id to eos_token_id: {processor.tokenizer.eos_token_id}")
except Exception as e_proc:
    print(f"Error loading Whisper processor: {e_proc}")
    raise
print("-" * 30)

# 3. Prepare Dataset for Whisper (with Augmentation)
print("--- Preparing Dataset for Whisper ---")
time_augment_pipeline = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.010, p=0.3),
    TimeStretch(min_rate=0.85, max_rate=1.15, p=0.3, leave_length_unchanged=False),
    PitchShift(min_semitones=-3, max_semitones=3, p=0.3)
])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
spec_augment_probability = 0.5

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            print(f"Error in load_process_audio_batch for {audio_path}: {e_lp}")
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
try:
    for split in raw_datasets:
        original_len = len(raw_datasets[split])
        raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")
        if len(raw_datasets[split]) != original_len:
            print(f"Filtered out {original_len - len(raw_datasets[split])} missing paths from {split} split.")
        if len(raw_datasets[split]) == 0 and split == "train": raise ValueError(f"Train split empty after path filtering.")
    print(f"Train size after path filter: {len(raw_datasets['train'])}")
except Exception as e_filt:
    print(f"Path filtering error: {e_filt}")
    raise

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    print(f"Processing {split_name} audio (augment={is_train})...")
    if len(dataset_split) == 0:
        print(f"Skipping processing for empty split: {split_name}")
        processed_splits[split_name] = dataset_split
        continue
    try:
        processed_splits[split_name] = dataset_split.map(
            lambda batch: load_process_audio_batch(batch, do_augment=is_train),
            batched=True, batch_size=8, num_proc=1,
            desc=f"Processing {split_name} audio"
        )
        print(f"{split_name} size before input_features filter: {len(processed_splits[split_name])}")
        processed_splits[split_name] = processed_splits[split_name].filter(
            lambda example: example["input_features"] is not None,
            num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
        )
        print(f"{split_name} size after input_features filter: {len(processed_splits[split_name])}")
        if len(processed_splits[split_name]) == 0 and split_name == "train":
            raise ValueError("Training dataset empty after input_features filter.")
    except Exception as e_proc_split:
        print(f"Error processing {split_name}: {e_proc_split}")
        raise
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()
print("Audio features processed.")
if len(raw_datasets_with_features.get("train", [])) > 0:
    print(f"Columns after feature extraction: {raw_datasets_with_features['train'].features}")

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
try:
    processed_datasets = raw_datasets_with_features.map(
        prepare_dataset_labels, batched=True, batch_size=8,
        remove_columns=["audio_path", "sentence"],
        num_proc=1, desc="Tokenizing text"
    )
    del raw_datasets_with_features; gc.collect()
except Exception as e_tok:
    print(f"Tokenization error: {e_tok}")
    raise

for split_name in processed_datasets:
    print(f"{split_name} size before final combined filter: {len(processed_datasets[split_name])}")
    if len(processed_datasets[split_name]) == 0: continue
    try:
        processed_datasets[split_name] = processed_datasets[split_name].filter(
            lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                       ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
            num_proc=1, desc=f"Final filtering on {split_name}"
        )
        print(f"{split_name} size after final combined filter: {len(processed_datasets[split_name])}")
        if len(processed_datasets[split_name]) == 0 and split_name == "train":
            raise ValueError("Training dataset empty after final combined filter.")
    except Exception as e_filt_final:
        print(f"Final filtering error on {split_name}: {e_filt_final}")
        raise
print("Dataset prepared for Whisper training.")
if len(processed_datasets.get("train",[])) > 0:
    print(f"Processed dataset final columns: {list(processed_datasets['train'].features.keys())}")
    print(f"Sample input_features type: {type(processed_datasets['train'][0]['input_features'])}")
    sample_feat_np = np.asarray(processed_datasets['train'][0]['input_features'])
    print(f"Sample input_features shape: {sample_feat_np.shape}")
    print(f"Sample labels (first 10): {processed_datasets['train'][0]['labels'][:10]}")

print(f"Dataset sizes - Train: {len(processed_datasets.get('train',[]))}, Val: {len(processed_datasets.get('val',[]))}, Test: {len(processed_datasets.get('test',[]))}")
print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
print("--- Defining Seq2Seq Data Collator ---")
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Filter out features where input_features or labels might be None or labels empty
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {} # Return empty dict if no valid features after filtering

        input_features_list = [f["input_features"] for f in valid_features]
        # Pad the input features
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        # Tokenize and pad the label features
        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # If bos token is appendend in previous tokenization, slice it
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data Collator defined.")
print("-" * 30)

# 5. Define Evaluation Metrics
print("--- Defining Evaluation Metrics ---")
metric = evaluate.load("wer")

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r'[^\w\s\']', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # Decode predictions and labels
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    # Normalize text
    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    # Print a few examples for qualitative inspection
    for i in range(min(3, len(pred_str))):
        print(f"Pred [{i}]: {pred_str[i]}")
        print(f"Ref  [{i}]: {label_str[i]}")

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    # Log to WandB
    try:
        # Check if trainer object exists and has state
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer}) # Log WER without step if trainer state not available
    except Exception as e_wandb_log:
        print(f"WandB logging error in compute_metrics: {e_wandb_log}")

    return {"wer": wer}
print("Compute Metrics function defined with WandB logging.")
print("-" * 30)

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
try:
    model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
    model.config.suppress_tokens = []
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    if torch.cuda.is_available(): model.to("cuda")
    print("Whisper model loaded and LoRA configured.")
    print(f"Model is on device: {next(model.parameters()).device}")
except Exception as e_model:
    print(f"Error loading model or applying LoRA: {e_model}")
    raise
print("Checking GPU memory after model loading...")
!nvidia-smi
print("-" * 30)

#  7. Define Training Arguments
print("--- Defining Training Arguments for Seq2Seq + LoRA ---")
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,

)
print("\nTraining Arguments:")
print(f" -> LR: {training_args.learning_rate}, Max Epochs: {training_args.num_train_epochs}")
eff_bs = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * (torch.cuda.device_count() if torch.cuda.is_available() else 1)
print(f" -> Eff. BS: {eff_bs}, FP16: {training_args.fp16}")
print(f" -> Generation Num Beams for Eval: {training_args.generation_num_beams}")
print(f" -> Early Stopping will be handled by a callback. Weight Decay: {training_args.weight_decay}")
print(f" -> Output Dir: {training_args.output_dir}")
print(f" -> Logging to WandB enabled.")
print("-" * 30)

# 8. Initialize Trainer
print("--- Initializing Seq2Seq Trainer ---")
if len(processed_datasets.get("train", [])) == 0: raise ValueError("Training dataset is empty. Cannot initialize trainer.")
if len(processed_datasets.get("val", [])) == 0: raise ValueError("Validation dataset is empty. Cannot initialize trainer for evaluation.")

early_stopping_patience_value =
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)
print(f"Early stopping enabled with patience: {early_stopping_patience_value} evaluations.")

try:
    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=processed_datasets["train"],
        eval_dataset=processed_datasets["val"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor,
        callbacks=[early_stopping_callback]
    )
    print("Trainer initialized.")
except Exception as e_trainer:
    print(f"Error initializing trainer: {e_trainer}")
    raise
print("-" * 30)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA and Augmentation ---")
final_trainer_state = None
try:
    print("Starting training from scratch with the new LoRA configuration (q,v,k,out + fc1,fc2) and WandB logging...")
    torch.cuda.empty_cache()
    gc.collect()
    print("Checking GPU memory before training...")
    !nvidia-smi
    train_result = trainer.train()
    final_trainer_state = trainer.state
    print("\nTraining finished.")
    print(f"Train metrics: {train_result.metrics}")
    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)
    print(f"Training metrics saved to {metrics_path}")

    print("\nSaving final PEFT adapter model (best model based on 'wer')...")
    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    print(f"✅ Final PEFT adapter (final state) saved to {os.path.join(output_dir, 'final_adapter')}")
    processor.save_pretrained(output_dir)
    print(f"✅ Processor saved to {output_dir}")

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None}) # Log best eval WER

except RuntimeError as e_rt:
    if "out of memory" in str(e_rt).lower():
        print("\nCUDA OOM Error! Try reducing LoRA rank 'r', effective batch_size, or target_modules, or using a smaller base model.")
    print(f"Runtime Error during training: {e_rt}")
    !nvidia-smi
    raise
except Exception as e_tr:
    print(f"\nTraining error: {type(e_tr).__name__} - {e_tr}")
    !nvidia-smi
    raise

# Final evaluation using the best model loaded by the Trainer
if final_trainer_state and trainer.state.best_model_checkpoint:
    print(f"\nEvaluating on validation set using the best model from {trainer.state.best_model_checkpoint}...")
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        print(f"Final Evaluation results (best model): {eval_results}")
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        print(f"Evaluation metrics for best model saved to {eval_metrics_path}")
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")
else:
    print("\nSkipping final evaluation as training did not complete successfully or no best model checkpoint found.")
print("-" * 30)

# 10. Generate Transcriptions on Test Set
print("--- Generating Transcriptions for Test Set using PEFT Model ---")
# Use the best model checkpoint identified by the trainer for inference
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_aug.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    print(f"Loading best PEFT adapter from: {best_checkpoint_dir} for inference...")
    try:
        base_model_name_or_path = model_name_loaded
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32

        base_model = WhisperForConditionalGeneration.from_pretrained(
            base_model_name_or_path, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()

        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir) # Load processor from checkpoint
        print("Best PEFT model and processor loaded successfully for inference.")
    except Exception as e_load:
        print(f"Error loading best PEFT model for inference from {best_checkpoint_dir}: {e_load}.")
else:
    print(f"Best model checkpoint not found or training didn't complete. Check: {best_checkpoint_dir}")


def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]):
        print("Skipping transcription generation (model/processor not loaded or load failed).")
        return
    results = []
    device = inference_model.device
    print(f"Generating transcriptions for {len(test_dataset)} samples on device: {device}...")

    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            # Duration calculation (rough estimate based on features)
            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            trans = f"Error: {type(e_inf).__name__} - {str(e_inf)[:100]}"
            print(f"Inference error for one sample: {e_inf}")
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        print(f"✅ Transcriptions saved to {output_csv_path}")
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        # Calculate WER on test set
        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        print(f"Error saving transcriptions CSV or calculating test WER: {e_csv}")

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)
else:
    print("Skipping inference generation on test set (model/processor load failed or test set empty).")
print("-" * 30)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded. Training may have failed or was skipped."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        print(f"Gradio input: {audio_filepath}")
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        print(f" -> Gradio Output: {transcription}")
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        print(f"Gradio Error: {type(e_grad).__name__} - {e_grad}")
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA + Aug)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA (r={lora_config.r}, alpha={lora_config.lora_alpha}, targets={lora_config.target_modules}) & Augmentation. Dataset: {wandb.config.num_samples_train} train samples.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        print(f"Error launching Gradio: {e_launch}.")
else:
    print("\nSkipping Gradio launch (model/processor not loaded, possibly due to issues in training or loading).")
print("\n--- Script Finished ---")
wandb.finish()

In [ ]:
"""
Standalone Gradio Deployment Script for Swahili ASR Whisper+LoRA Model.

This script loads a fine-tuned PEFT (LoRA) adapter onto the base Whisper model
and launches a Gradio web interface for live audio transcription.

"""
import os
import torch
import gradio as gr
import torchaudio
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)
from peft import PeftModel


PEFT_ADAPTER_PATH = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/checkpoint-3000"
BASE_MODEL_NAME = "openai/whisper-small"

# Parameters for transcription
LANGUAGE_ABBR = "sw"
TASK = "transcribe"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

# Model Loading
def load_model(adapter_path):
    """
    Loads the base Whisper model, applies the fine-tuned PEFT adapter,
    and loads the appropriate processor.
    """
    print(f"Loading base model: {BASE_MODEL_NAME}...")
    try:
        # 1. Load the base model
        base_model = WhisperForConditionalGeneration.from_pretrained(
            BASE_MODEL_NAME,
            torch_dtype=MODEL_DTYPE
        ).to(DEVICE)

        print(f"Loading PEFT adapter from: {adapter_path}...")
        # 2. Load and apply the LoRA adapter
        inference_model = PeftModel.from_pretrained(base_model, adapter_path)
        inference_model.eval() # Set the model to evaluation mode

        # 3. Load the processor from the base model name
        print(f"Loading processor from base model: {BASE_MODEL_NAME}...")
        processor = WhisperProcessor.from_pretrained(
            BASE_MODEL_NAME,
            language="Swahili",
            task="transcribe"
        )

        print("✅ Model and processor loaded successfully.")
        return inference_model, processor

    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("Please ensure the 'PEFT_ADAPTER_PATH' is correct and you have necessary permissions.")
        return None, None

#  Main Application Logic
print("--- Initializing Swahili ASR Gradio App ---")
model, processor = load_model(PEFT_ADAPTER_PATH)

def transcribe_audio(audio_filepath):
    """
    The main function to be called by Gradio for transcribing audio.
    """
    if not model or not processor:
        return "Error: Model is not loaded. Please check the console for errors."

    if not audio_filepath or not os.path.exists(audio_filepath):
        return "Error: Audio file is missing or path is invalid. Please upload a file."

    try:
        # Load and process the audio file
        waveform, sampling_rate = torchaudio.load(audio_filepath)

        # Ensure mono channel and 16kHz sampling rate
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sampling_rate != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
            waveform = resampler(waveform)

        # Extract features and move to the correct device
        input_features = processor(
            waveform.squeeze().numpy(),
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.to(DEVICE).to(MODEL_DTYPE)

        # Generate token IDs
        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                forced_decoder_ids=processor.get_decoder_prompt_ids(language=LANGUAGE_ABBR, task=TASK),
                num_beams=5,
                max_new_tokens=256
            )[0]

        # Decode the token IDs to text
        transcription = processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)

        print(f"Input: {os.path.basename(audio_filepath)} -> Transcription: {transcription}")
        return transcription

    except Exception as e:
        print(f"An error occurred during transcription: {e}")
        return f"Transcription failed. Error: {type(e).__name__}."

# Gradio Interface
if model and processor:
    interface = gr.Interface(
        fn=transcribe_audio,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title="Swahili ASR with Fine-Tuned Whisper-Small + LoRA",
        description=(
            "An interactive demo for transcribing Swahili speech to text. "
            f"This app uses the '{BASE_MODEL_NAME}' model fine-tuned with a LoRA adapter "
            f"loaded from '{os.path.basename(PEFT_ADAPTER_PATH)}'."
        ),
        allow_flagging="never",
        examples=[]
    )

    print("\n🚀 Launching Gradio interface...")
    interface.launch(share=True, debug=False)
else:
    print("\nCould not launch Gradio interface because the model failed to load.")

LoRA with no augmentation" See below

In [ ]:
# CELL 1: RUN THIS FIRST, THEN RESTART RUNTIME
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2

# DOWNGRADED TO CUDA 12.1 (+cu121) TO MATCH COLAB DRIVERS
!pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121

!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.44.1
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb

In [ ]:
# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + Baseline/No Augmentation + WandB) ---")

# Disable frozen modules and debugging validations
import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

try:
    import json
    import re
    import numpy as np
    import pandas as pd
    import torch
    import torchvision
    import torchaudio
    from torchaudio.transforms import SpecAugment
    import transformers
    import datasets
    import accelerate
    import evaluate
    import wandb
    from tqdm.notebook import tqdm
    from sklearn.model_selection import train_test_split
    from datasets import Dataset, DatasetDict
    from transformers import (
        WhisperProcessor,
        WhisperFeatureExtractor,
        WhisperTokenizerFast,
        WhisperForConditionalGeneration,
        Seq2SeqTrainingArguments,
        Seq2SeqTrainer,
        EarlyStoppingCallback
    )
    from peft import (
        prepare_model_for_kbit_training,
        LoraConfig,
        PeftModel,
        PeftConfig,
        get_peft_model
    )
    from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
    from dataclasses import dataclass
    from typing import Any, Dict, List, Union
    import warnings
    import gradio as gr
    import csv
    import soundfile as sf
    import gc
    warnings.filterwarnings("ignore")
except ImportError as e:
    print(f"Import error: {e}")
    raise

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Torchaudio version: {torchaudio.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"Accelerate version: {accelerate.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"WandB version: {wandb.__version__}")
try:
    import peft
    print(f"PEFT version: {peft.__version__}")
    import audiomentations
    print(f"Audiomentations version: {audiomentations.__version__}")
except ImportError as e:
    print(f"Failed to import library: {e}")
    raise ImportError("Required library missing. Check installation logs.")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Device Count: {torch.cuda.device_count()}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-baseline-no-aug", # UPDATED RUN NAME
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "None", # UPDATED CONFIG
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
print(f"CSV found at: {csv_path}")
print(f"Audio base directory: {audio_base_path}")
print(f"Model outputs to: {output_dir}")
os.makedirs(output_dir, exist_ok=True)

try:
    df = pd.read_csv(csv_path).head(9000)
    print(f"Initial dataframe shape: {df.shape}")
except FileNotFoundError:
    print(f"Error loading CSV from {csv_path}. Upload swahili_dataset.csv to /content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/")
    raise

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text) # Keeps apostrophes
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]
print(f"Shape after cleaning: {df.shape}")

try:
    print("Skipping audio validation to save time. Assuming all audio files are valid.")
    if len(df) == 0: raise ValueError("No valid samples after transcript cleaning.")
    # Ensuring 80/10/10 split on 9000 samples -> 7200 train, 900 val, 900 test
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True) # 0.2 * 9000 = 1800 for temp
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True) # 0.5 * 1800 = 900 each
    del df, temp_df; gc.collect()
except Exception as e_split:
    print(f"Dataset splitting error: {e_split}")
    raise

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"\nDataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")
if len(raw_datasets["train"]) > 0: print(f"Sample 0 (raw): {raw_datasets['train'][0]}")
print(f"Raw dataset columns: {raw_datasets['train'].features}")
print("-" * 30)

model_name_loaded = "openai/whisper-small" #model being loaded
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
try:
    feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
    tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
    processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
    print(f"Whisper Processor for {language} ({language_abbr}), task: {task} loaded from {model_name_loaded}.")
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id
        print(f"Set processor.tokenizer.pad_token_id to eos_token_id: {processor.tokenizer.eos_token_id}")
except Exception as e_proc:
    print(f"Error loading Whisper processor: {e_proc}")
    raise
print("-" * 30)

# 3. Prepare Dataset for Whisper (BASELINE - NO AUGMENTATION)
print("--- Preparing Dataset for Whisper (Baseline) ---")

# EMPTY THE TIME-DOMAIN PIPELINE
time_augment_pipeline = Compose([])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
# SET SPECAUGMENT PROBABILITY TO 0.0
spec_augment_probability = 0.0

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            print(f"Error in load_process_audio_batch for {audio_path}: {e_lp}")
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
try:
    for split in raw_datasets:
        original_len = len(raw_datasets[split])
        raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")
        if len(raw_datasets[split]) != original_len:
            print(f"Filtered out {original_len - len(raw_datasets[split])} missing paths from {split} split.")
        if len(raw_datasets[split]) == 0 and split == "train": raise ValueError(f"Train split empty after path filtering.")
    print(f"Train size after path filter: {len(raw_datasets['train'])}")
except Exception as e_filt:
    print(f"Path filtering error: {e_filt}")
    raise

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    print(f"Processing {split_name} audio (augment={is_train})...")
    if len(dataset_split) == 0:
        print(f"Skipping processing for empty split: {split_name}")
        processed_splits[split_name] = dataset_split
        continue
    try:
        processed_splits[split_name] = dataset_split.map(
            lambda batch: load_process_audio_batch(batch, do_augment=is_train),
            batched=True, batch_size=8, num_proc=1, # num_proc=1 for Colab stability with audio
            desc=f"Processing {split_name} audio"
        )
        print(f"{split_name} size before input_features filter: {len(processed_splits[split_name])}")
        processed_splits[split_name] = processed_splits[split_name].filter(
            lambda example: example["input_features"] is not None,
            num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
        )
        print(f"{split_name} size after input_features filter: {len(processed_splits[split_name])}")
        if len(processed_splits[split_name]) == 0 and split_name == "train":
            raise ValueError("Training dataset empty after input_features filter.")
    except Exception as e_proc_split:
        print(f"Error processing {split_name}: {e_proc_split}")
        raise
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()
print("Audio features processed.")
if len(raw_datasets_with_features.get("train", [])) > 0:
    print(f"Columns after feature extraction: {raw_datasets_with_features['train'].features}")

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
try:
    processed_datasets = raw_datasets_with_features.map(
        prepare_dataset_labels, batched=True, batch_size=8,
        remove_columns=["audio_path", "sentence"],
        num_proc=1, desc="Tokenizing text"
    )
    del raw_datasets_with_features; gc.collect()
except Exception as e_tok:
    print(f"Tokenization error: {e_tok}")
    raise

for split_name in processed_datasets:
    print(f"{split_name} size before final combined filter: {len(processed_datasets[split_name])}")
    if len(processed_datasets[split_name]) == 0: continue
    try:
        processed_datasets[split_name] = processed_datasets[split_name].filter(
            lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                       ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
            num_proc=1, desc=f"Final filtering on {split_name}"
        )
        print(f"{split_name} size after final combined filter: {len(processed_datasets[split_name])}")
        if len(processed_datasets[split_name]) == 0 and split_name == "train":
            raise ValueError("Training dataset empty after final combined filter.")
    except Exception as e_filt_final:
        print(f"Final filtering error on {split_name}: {e_filt_final}")
        raise
print("Dataset prepared for Whisper training.")
if len(processed_datasets.get("train",[])) > 0:
    print(f"Processed dataset final columns: {list(processed_datasets['train'].features.keys())}")
    print(f"Sample input_features type: {type(processed_datasets['train'][0]['input_features'])}")
    sample_feat_np = np.asarray(processed_datasets['train'][0]['input_features'])
    print(f"Sample input_features shape: {sample_feat_np.shape}")
    print(f"Sample labels (first 10): {processed_datasets['train'][0]['labels'][:10]}")

print(f"Dataset sizes - Train: {len(processed_datasets.get('train',[]))}, Val: {len(processed_datasets.get('val',[]))}, Test: {len(processed_datasets.get('test',[]))}")
print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
print("--- Defining Seq2Seq Data Collator ---")
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Filter out features where input_features or labels might be None or labels empty
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {} # Return empty dict if no valid features after filtering

        input_features_list = [f["input_features"] for f in valid_features]
        # Pad the input features
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        # Tokenize and pad the label features
        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # If bos token is appendend in previous tokenization, slice it
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data Collator defined.")
print("-" * 30)

# 5. Define Evaluation Metrics
print("--- Defining Evaluation Metrics ---")
metric = evaluate.load("wer")

def normalize_text(text): # Ensure consistent normalization
    if not isinstance(text, str): text = str(text) # Handle potential non-string inputs
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text) # Keeps apostrophes
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # Decode predictions and labels
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    # Normalize text
    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    # Print a few examples for qualitative inspection
    for i in range(min(3, len(pred_str))): # Print up to 3 examples
        print(f"Pred [{i}]: {pred_str[i]}")
        print(f"Ref  [{i}]: {label_str[i]}")

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    # Log to WandB
    try:
        # Check if trainer object exists and has state
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer}) # Log WER without step if trainer state not available
    except Exception as e_wandb_log:
        print(f"WandB logging error in compute_metrics: {e_wandb_log}")

    return {"wer": wer}
print("Compute Metrics function defined with WandB logging.")
print("-" * 30)

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
try:
    model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
    model.config.suppress_tokens = []
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    if torch.cuda.is_available(): model.to("cuda")
    print("Whisper model loaded and LoRA configured.")
    print(f"Model is on device: {next(model.parameters()).device}")
except Exception as e_model:
    print(f"Error loading model or applying LoRA: {e_model}")
    raise
print("Checking GPU memory after model loading...")
!nvidia-smi
print("-" * 30)

#  7. Define Training Arguments
print("--- Defining Training Arguments for Seq2Seq + LoRA ---")
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)
print("\nTraining Arguments:")
print(f" -> LR: {training_args.learning_rate}, Max Epochs: {training_args.num_train_epochs}")
eff_bs = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * (torch.cuda.device_count() if torch.cuda.is_available() else 1)
print(f" -> Eff. BS: {eff_bs}, FP16: {training_args.fp16}")
print(f" -> Generation Num Beams for Eval: {training_args.generation_num_beams}")
print(f" -> Early Stopping will be handled by a callback. Weight Decay: {training_args.weight_decay}")
print(f" -> Output Dir: {training_args.output_dir}")
print(f" -> Logging to WandB enabled.")
print("-" * 30)

# 8. Initialize Trainer
print("--- Initializing Seq2Seq Trainer ---")
if len(processed_datasets.get("train", [])) == 0: raise ValueError("Training dataset is empty. Cannot initialize trainer.")
if len(processed_datasets.get("val", [])) == 0: raise ValueError("Validation dataset is empty. Cannot initialize trainer for evaluation.")

early_stopping_patience_value = 5 # Ensure this value is properly set
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)
print(f"Early stopping enabled with patience: {early_stopping_patience_value} evaluations.")

try:
    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=processed_datasets["train"],
        eval_dataset=processed_datasets["val"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor,
        callbacks=[early_stopping_callback]
    )
    print("Trainer initialized.")
except Exception as e_trainer:
    print(f"Error initializing trainer: {e_trainer}")
    raise
print("-" * 30)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (Baseline) ---")
final_trainer_state = None
try:
    print("Starting training from scratch with baseline LoRA configuration and WandB logging...")
    torch.cuda.empty_cache()
    gc.collect()
    print("Checking GPU memory before training...")
    !nvidia-smi
    train_result = trainer.train()
    final_trainer_state = trainer.state
    print("\nTraining finished.")
    print(f"Train metrics: {train_result.metrics}")
    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)
    print(f"Training metrics saved to {metrics_path}")

    print("\nSaving final PEFT adapter model (best model based on 'wer')...")
    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    print(f"✅ Final PEFT adapter (final state) saved to {os.path.join(output_dir, 'final_adapter')}")
    processor.save_pretrained(output_dir)
    print(f"✅ Processor saved to {output_dir}")

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None}) # Log best eval WER

except RuntimeError as e_rt:
    if "out of memory" in str(e_rt).lower():
        print("\nCUDA OOM Error! Try reducing LoRA rank 'r', effective batch_size, or target_modules, or using a smaller base model.")
    print(f"Runtime Error during training: {e_rt}")
    !nvidia-smi
    raise
except Exception as e_tr:
    print(f"\nTraining error: {type(e_tr).__name__} - {e_tr}")
    !nvidia-smi
    raise

# Final evaluation using the best model loaded by the Trainer
if final_trainer_state and trainer.state.best_model_checkpoint:
    print(f"\nEvaluating on validation set using the best model from {trainer.state.best_model_checkpoint}...")
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        print(f"Final Evaluation results (best model): {eval_results}")
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        print(f"Evaluation metrics for best model saved to {eval_metrics_path}")
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")
else:
    print("\nSkipping final evaluation as training did not complete successfully or no best model checkpoint found.")
print("-" * 30)

# 10. Generate Transcriptions on Test Set
print("--- Generating Transcriptions for Test Set using PEFT Model ---")
# Use the best model checkpoint identified by the trainer for inference
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_baseline.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    print(f"Loading best PEFT adapter from: {best_checkpoint_dir} for inference...")
    try:
        base_model_name_or_path = model_name_loaded
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32

        base_model = WhisperForConditionalGeneration.from_pretrained(
            base_model_name_or_path, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()

        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir) # Load processor from checkpoint
        print("Best PEFT model and processor loaded successfully for inference.")
    except Exception as e_load:
        print(f"Error loading best PEFT model for inference from {best_checkpoint_dir}: {e_load}.")
else:
    print(f"Best model checkpoint not found or training didn't complete. Check: {best_checkpoint_dir}")


def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]):
        print("Skipping transcription generation (model/processor not loaded or load failed).")
        return
    results = []
    device = inference_model.device
    print(f"Generating transcriptions for {len(test_dataset)} samples on device: {device}...")

    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            # Duration calculation (rough estimate based on features)
            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01 # Approx for Whisper features
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            trans = f"Error: {type(e_inf).__name__} - {str(e_inf)[:100]}"
            print(f"Inference error for one sample: {e_inf}")
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        print(f"✅ Transcriptions saved to {output_csv_path}")
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        # Calculate WER on test set
        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        print(f"Error saving transcriptions CSV or calculating test WER: {e_csv}")

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)
else:
    print("Skipping inference generation on test set (model/processor load failed or test set empty).")
print("-" * 30)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded. Training may have failed or was skipped."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        print(f"Gradio input: {audio_filepath}")
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        print(f" -> Gradio Output: {transcription}")
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        print(f"Gradio Error: {type(e_grad).__name__} - {e_grad}")
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA Base)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA (r={lora_config.r}, alpha={lora_config.lora_alpha}, targets={lora_config.target_modules}) & NO Augmentation. Dataset: {wandb.config.num_samples_train} train samples.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        print(f"Error launching Gradio: {e_launch}.")
else:
    print("\nSkipping Gradio launch (model/processor not loaded, possibly due to issues in training or loading).")
print("\n--- Script Finished ---")
wandb.finish()

2. LoRA + Gaussian Noise Only

In [ ]:
# CELL 1: RUN THIS FIRST, THEN RESTART RUNTIME
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2

# We rely on Colab's native PyTorch, but we MUST bump bitsandbytes to 0.45.2 to fix the triton.ops error
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb

In [ ]:
# CELL 2: RUN THIS AFTER RESTARTING

# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + Gaussian Noise Augmentation + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import pandas as pd
import torch
import torchvision
import torchaudio
from torchaudio.transforms import SpecAugment
import transformers
import datasets
import accelerate
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizerFast,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    PeftModel,
    PeftConfig,
    get_peft_model
)
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
import gradio as gr
import csv
import soundfile as sf
import gc
warnings.filterwarnings("ignore")

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-aug-gaussian-only", # UPDATED RUN NAME
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "GaussianNoise", # UPDATED CONFIG
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path).head(9000)

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

print("Skipping audio validation to save time. Assuming all audio files are valid.")
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"Dataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")

model_name_loaded = "openai/whisper-small"
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# 3. Prepare Dataset for Whisper (GAUSSIAN NOISE ONLY)
print("--- Preparing Dataset for Whisper (Gaussian Noise Only) ---")

# ONLY GAUSSIAN NOISE IN THE PIPELINE
time_augment_pipeline = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.010, p=1.0)
])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
# SET SPECAUGMENT PROBABILITY TO 0.0
spec_augment_probability = 0.0

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
for split in raw_datasets:
    raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    if len(dataset_split) == 0:
        processed_splits[split_name] = dataset_split
        continue
    processed_splits[split_name] = dataset_split.map(
        lambda batch: load_process_audio_batch(batch, do_augment=is_train),
        batched=True, batch_size=8, num_proc=1,
        desc=f"Processing {split_name} audio"
    )
    processed_splits[split_name] = processed_splits[split_name].filter(
        lambda example: example["input_features"] is not None,
        num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
    )
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
processed_datasets = raw_datasets_with_features.map(
    prepare_dataset_labels, batched=True, batch_size=8,
    remove_columns=["audio_path", "sentence"],
    num_proc=1, desc="Tokenizing text"
)
del raw_datasets_with_features; gc.collect()

for split_name in processed_datasets:
    processed_datasets[split_name] = processed_datasets[split_name].filter(
        lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                   ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
        num_proc=1, desc=f"Final filtering on {split_name}"
    )

print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {}

        input_features_list = [f["input_features"] for f in valid_features]
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 5. Define Evaluation Metrics
metric = evaluate.load("wer")

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    try:
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer})
    except Exception as e_wandb_log:
        pass

    return {"wer": wer}

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
model.config.suppress_tokens = []
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available(): model.to("cuda")

#  7. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)

# 8. Initialize Trainer
early_stopping_patience_value = 5
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback]
)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (Gaussian Noise Only) ---")
final_trainer_state = None
try:
    torch.cuda.empty_cache()
    gc.collect()
    train_result = trainer.train()
    final_trainer_state = trainer.state

    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)

    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    processor.save_pretrained(output_dir)

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None})

except RuntimeError as e_rt:
    print(f"Runtime Error during training: {e_rt}")
    raise

# Final evaluation
if final_trainer_state and trainer.state.best_model_checkpoint:
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")

# 10. Generate Transcriptions on Test Set
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_gaussian.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    try:
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32
        base_model = WhisperForConditionalGeneration.from_pretrained(
            model_name_loaded, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()
        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir)
    except Exception as e_load:
        pass

def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]): return
    results = []
    device = inference_model.device
    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            pass
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        pass

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA Gaussian Aug)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA & Gaussian Noise Augmentation only.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        pass

print("\n--- Script Finished ---")
wandb.finish()

3. LoRA + Time Stretch Only

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 1: RUN THIS FIRST, THEN RESTART RUNTIME
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2

# We rely on Colab's native PyTorch.
# bitsandbytes is bumped to 0.45.2 to fix the triton.ops error
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb

In [ ]:
# CELL 2: RUN THIS AFTER RESTARTING

# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + Time Stretch Augmentation + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import pandas as pd
import torch
import torchvision
import torchaudio
from torchaudio.transforms import SpecAugment
import transformers
import datasets
import accelerate
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizerFast,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    PeftModel,
    PeftConfig,
    get_peft_model
)
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
import gradio as gr
import csv
import soundfile as sf
import gc
warnings.filterwarnings("ignore")

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-aug-timestretch-only", # UPDATED RUN NAME
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "TimeStretch", # UPDATED CONFIG
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path).head(9000)

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

print("Skipping audio validation to save time. Assuming all audio files are valid.")
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"Dataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")

model_name_loaded = "openai/whisper-small"
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# 3. Prepare Dataset for Whisper (TIME STRETCH ONLY)
print("--- Preparing Dataset for Whisper (Time Stretch Only) ---")

# ONLY TIME STRETCH IN THE PIPELINE (p=1.0 to ensure it applies when do_augment=True)
time_augment_pipeline = Compose([
    TimeStretch(min_rate=0.85, max_rate=1.15, p=1.0, leave_length_unchanged=False)
])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
# SET SPECAUGMENT PROBABILITY TO 0.0
spec_augment_probability = 0.0

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
for split in raw_datasets:
    raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    if len(dataset_split) == 0:
        processed_splits[split_name] = dataset_split
        continue
    processed_splits[split_name] = dataset_split.map(
        lambda batch: load_process_audio_batch(batch, do_augment=is_train),
        batched=True, batch_size=8, num_proc=1,
        desc=f"Processing {split_name} audio"
    )
    processed_splits[split_name] = processed_splits[split_name].filter(
        lambda example: example["input_features"] is not None,
        num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
    )
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
processed_datasets = raw_datasets_with_features.map(
    prepare_dataset_labels, batched=True, batch_size=8,
    remove_columns=["audio_path", "sentence"],
    num_proc=1, desc="Tokenizing text"
)
del raw_datasets_with_features; gc.collect()

for split_name in processed_datasets:
    processed_datasets[split_name] = processed_datasets[split_name].filter(
        lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                   ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
        num_proc=1, desc=f"Final filtering on {split_name}"
    )

print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {}

        input_features_list = [f["input_features"] for f in valid_features]
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 5. Define Evaluation Metrics
metric = evaluate.load("wer")

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    try:
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer})
    except Exception as e_wandb_log:
        pass

    return {"wer": wer}

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
model.config.suppress_tokens = []
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available(): model.to("cuda")

#  7. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)

# 8. Initialize Trainer
early_stopping_patience_value = 5
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback]
)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (Time Stretch Only) ---")
final_trainer_state = None
try:
    torch.cuda.empty_cache()
    gc.collect()
    train_result = trainer.train()
    final_trainer_state = trainer.state

    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)

    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    processor.save_pretrained(output_dir)

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None})

except RuntimeError as e_rt:
    print(f"Runtime Error during training: {e_rt}")
    raise

# Final evaluation
if final_trainer_state and trainer.state.best_model_checkpoint:
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")

# 10. Generate Transcriptions on Test Set
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_timestretch.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    try:
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32
        base_model = WhisperForConditionalGeneration.from_pretrained(
            model_name_loaded, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()
        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir)
    except Exception as e_load:
        pass

def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]): return
    results = []
    device = inference_model.device
    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            pass
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        pass

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA Time Stretch Aug)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA & Time Stretch Augmentation only.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        pass

print("\n--- Script Finished ---")
wandb.finish()

4. LoRA + Pitch Shift Only

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 1: RUN THIS FIRST, THEN RESTART RUNTIME
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2

# We rely on Colab's native PyTorch.
# bitsandbytes is bumped to 0.45.2 to fix the triton.ops error
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb

In [ ]:
# CELL 1: RUN THIS FIRST, THEN RESTART RUNTIME
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2

# We rely on Colab's native PyTorch.
# bitsandbytes is bumped to 0.45.2 to fix the triton.ops error
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb

In [ ]:
# CELL 2: RUN THIS AFTER RESTARTING (OR DIRECTLY IF RUNTIME IS ALREADY ACTIVE)

# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + Pitch Shift Augmentation + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import pandas as pd
import torch
import torchvision
import torchaudio
from torchaudio.transforms import SpecAugment
import transformers
import datasets
import accelerate
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizerFast,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    PeftModel,
    PeftConfig,
    get_peft_model
)
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
import gradio as gr
import csv
import soundfile as sf
import gc
warnings.filterwarnings("ignore")

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-aug-pitchshift-only", # UPDATED RUN NAME
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "PitchShift", # UPDATED CONFIG
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path).head(9000)

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

print("Skipping audio validation to save time. Assuming all audio files are valid.")
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"Dataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")

model_name_loaded = "openai/whisper-small"
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# 3. Prepare Dataset for Whisper (PITCH SHIFT ONLY)
print("--- Preparing Dataset for Whisper (Pitch Shift Only) ---")

# ONLY PITCH SHIFT IN THE PIPELINE (p=1.0)
time_augment_pipeline = Compose([
    PitchShift(min_semitones=-3, max_semitones=3, p=1.0)
])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
# SET SPECAUGMENT PROBABILITY TO 0.0
spec_augment_probability = 0.0

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
for split in raw_datasets:
    raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    if len(dataset_split) == 0:
        processed_splits[split_name] = dataset_split
        continue
    processed_splits[split_name] = dataset_split.map(
        lambda batch: load_process_audio_batch(batch, do_augment=is_train),
        batched=True, batch_size=8, num_proc=1,
        desc=f"Processing {split_name} audio"
    )
    processed_splits[split_name] = processed_splits[split_name].filter(
        lambda example: example["input_features"] is not None,
        num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
    )
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
processed_datasets = raw_datasets_with_features.map(
    prepare_dataset_labels, batched=True, batch_size=8,
    remove_columns=["audio_path", "sentence"],
    num_proc=1, desc="Tokenizing text"
)
del raw_datasets_with_features; gc.collect()

for split_name in processed_datasets:
    processed_datasets[split_name] = processed_datasets[split_name].filter(
        lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                   ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
        num_proc=1, desc=f"Final filtering on {split_name}"
    )

print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {}

        input_features_list = [f["input_features"] for f in valid_features]
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 5. Define Evaluation Metrics
metric = evaluate.load("wer")

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    try:
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer})
    except Exception as e_wandb_log:
        pass

    return {"wer": wer}

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
model.config.suppress_tokens = []
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available(): model.to("cuda")

#  7. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)

# 8. Initialize Trainer
early_stopping_patience_value = 5
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback]
)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (Pitch Shift Only) ---")
final_trainer_state = None
try:
    torch.cuda.empty_cache()
    gc.collect()
    train_result = trainer.train()
    final_trainer_state = trainer.state

    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)

    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    processor.save_pretrained(output_dir)

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None})

except RuntimeError as e_rt:
    print(f"Runtime Error during training: {e_rt}")
    raise

# Final evaluation
if final_trainer_state and trainer.state.best_model_checkpoint:
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")

# 10. Generate Transcriptions on Test Set
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_pitchshift.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    try:
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32
        base_model = WhisperForConditionalGeneration.from_pretrained(
            model_name_loaded, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()
        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir)
    except Exception as e_load:
        pass

def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]): return
    results = []
    device = inference_model.device
    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            pass
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        pass

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA Pitch Shift Aug)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA & Pitch Shift Augmentation only.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        pass

print("\n--- Script Finished ---")
wandb.finish()+9+/9

5. LoRA + SpecAugment Only

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 1: RUN THIS FIRST, THEN RESTART RUNTIME
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2

# We rely on Colab's native PyTorch.
# bitsandbytes is bumped to 0.45.2 to fix the triton.ops error
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb

In [ ]:
# CELL 2: RUN THIS AFTER RESTARTING (OR DIRECTLY IF RUNTIME IS ALREADY ACTIVE)

# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + SpecAugment Only + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import pandas as pd
import torch
import torchvision
import torchaudio
from torchaudio.transforms import SpecAugment
import transformers
import datasets
import accelerate
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizerFast,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    PeftModel,
    PeftConfig,
    get_peft_model
)
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
import gradio as gr
import csv
import soundfile as sf
import gc
warnings.filterwarnings("ignore")

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-aug-specaugment-only", # UPDATED RUN NAME
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "SpecAugment", # UPDATED CONFIG
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path).head(9000)

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

print("Skipping audio validation to save time. Assuming all audio files are valid.")
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"Dataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")

model_name_loaded = "openai/whisper-small"
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# 3. Prepare Dataset for Whisper (SPECAUGMENT ONLY)
print("--- Preparing Dataset for Whisper (SpecAugment Only) ---")

# EMPTY THE TIME-DOMAIN PIPELINE
time_augment_pipeline = Compose([])

# KEEP SPECAUGMENT ACTIVE
spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
# SET SPECAUGMENT PROBABILITY TO 0.5 (50% chance of application during training)
spec_augment_probability = 0.5

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            # APPLY SPECAUGMENT ON THE FREQUENCY/TIME DOMAIN
            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
for split in raw_datasets:
    raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    if len(dataset_split) == 0:
        processed_splits[split_name] = dataset_split
        continue
    processed_splits[split_name] = dataset_split.map(
        lambda batch: load_process_audio_batch(batch, do_augment=is_train),
        batched=True, batch_size=8, num_proc=1,
        desc=f"Processing {split_name} audio"
    )
    processed_splits[split_name] = processed_splits[split_name].filter(
        lambda example: example["input_features"] is not None,
        num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
    )
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
processed_datasets = raw_datasets_with_features.map(
    prepare_dataset_labels, batched=True, batch_size=8,
    remove_columns=["audio_path", "sentence"],
    num_proc=1, desc="Tokenizing text"
)
del raw_datasets_with_features; gc.collect()

for split_name in processed_datasets:
    processed_datasets[split_name] = processed_datasets[split_name].filter(
        lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                   ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
        num_proc=1, desc=f"Final filtering on {split_name}"
    )

print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {}

        input_features_list = [f["input_features"] for f in valid_features]
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 5. Define Evaluation Metrics
metric = evaluate.load("wer")

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    try:
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer})
    except Exception as e_wandb_log:
        pass

    return {"wer": wer}

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
model.config.suppress_tokens = []
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available(): model.to("cuda")

#  7. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)

# 8. Initialize Trainer
early_stopping_patience_value = 5
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback]
)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (SpecAugment Only) ---")
final_trainer_state = None
try:
    torch.cuda.empty_cache()
    gc.collect()
    train_result = trainer.train()
    final_trainer_state = trainer.state

    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)

    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    processor.save_pretrained(output_dir)

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None})

except RuntimeError as e_rt:
    print(f"Runtime Error during training: {e_rt}")
    raise

# Final evaluation
if final_trainer_state and trainer.state.best_model_checkpoint:
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")

# 10. Generate Transcriptions on Test Set
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_specaugment.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    try:
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32
        base_model = WhisperForConditionalGeneration.from_pretrained(
            model_name_loaded, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()
        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir)
    except Exception as e_load:
        pass

def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]): return
    results = []
    device = inference_model.device
    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            pass
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        pass

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA SpecAugment Only)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA & SpecAugment Only.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        pass

print("\n--- Script Finished ---")
wandb.finish()

6: LoRA + Gaussian Noise + Pitch + TimeStretch + SpecAugment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 1: RUN THIS FIRST, THEN RESTART RUNTIME
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2

# We rely on Colab's native PyTorch.
# bitsandbytes is bumped to 0.45.2 to fix the triton.ops error
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb


In [ ]:
# CELL 2: RUN THIS AFTER RESTARTING (OR DIRECTLY IF RUNTIME IS ALREADY ACTIVE)

# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + ALL Augmentations + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import pandas as pd
import torch
import torchvision
import torchaudio
from torchaudio.transforms import SpecAugment
import transformers
import datasets
import accelerate
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizerFast,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    PeftModel,
    PeftConfig,
    get_peft_model
)
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
import gradio as gr
import csv
import soundfile as sf
import gc
warnings.filterwarnings("ignore")

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-aug-all-combined", # UPDATED RUN NAME
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "GaussianNoise+TimeStretch+PitchShift+SpecAugment", # UPDATED CONFIG
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path).head(9000)

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

print("Skipping audio validation to save time. Assuming all audio files are valid.")
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"Dataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")

model_name_loaded = "openai/whisper-small"
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# 3. Prepare Dataset for Whisper (ALL AUGMENTATIONS COMBINED)
print("--- Preparing Dataset for Whisper (All Augmentations Combined) ---")

time_augment_pipeline = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.010, p=0.3),
    TimeStretch(min_rate=0.85, max_rate=1.15, p=0.3, leave_length_unchanged=False),
    PitchShift(min_semitones=-3, max_semitones=3, p=0.3)
])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
spec_augment_probability = 0.5

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
for split in raw_datasets:
    raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    if len(dataset_split) == 0:
        processed_splits[split_name] = dataset_split
        continue
    processed_splits[split_name] = dataset_split.map(
        lambda batch: load_process_audio_batch(batch, do_augment=is_train),
        batched=True, batch_size=8, num_proc=1,
        desc=f"Processing {split_name} audio"
    )
    processed_splits[split_name] = processed_splits[split_name].filter(
        lambda example: example["input_features"] is not None,
        num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
    )
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
processed_datasets = raw_datasets_with_features.map(
    prepare_dataset_labels, batched=True, batch_size=8,
    remove_columns=["audio_path", "sentence"],
    num_proc=1, desc="Tokenizing text"
)
del raw_datasets_with_features; gc.collect()

for split_name in processed_datasets:
    processed_datasets[split_name] = processed_datasets[split_name].filter(
        lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                   ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
        num_proc=1, desc=f"Final filtering on {split_name}"
    )

print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {}

        input_features_list = [f["input_features"] for f in valid_features]
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 5. Define Evaluation Metrics
metric = evaluate.load("wer")

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    try:
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer})
    except Exception as e_wandb_log:
        pass

    return {"wer": wer}

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
model.config.suppress_tokens = []
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available(): model.to("cuda")

#  7. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)

# 8. Initialize Trainer
early_stopping_patience_value = 5
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback]
)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (All Augmentations Combined) ---")
final_trainer_state = None
try:
    torch.cuda.empty_cache()
    gc.collect()
    train_result = trainer.train()
    final_trainer_state = trainer.state

    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)

    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    processor.save_pretrained(output_dir)

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None})

except RuntimeError as e_rt:
    print(f"Runtime Error during training: {e_rt}")
    raise

# Final evaluation
if final_trainer_state and trainer.state.best_model_checkpoint:
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")

# 10. Generate Transcriptions on Test Set
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_all_aug.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    try:
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32
        base_model = WhisperForConditionalGeneration.from_pretrained(
            model_name_loaded, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()
        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir)
    except Exception as e_load:
        pass

def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]): return
    results = []
    device = inference_model.device
    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            pass
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        pass

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA + ALL Aug)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA & ALL Augmentations Combined (Gaussian, Time Stretch, Pitch Shift, SpecAugment).",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        pass

print("\n--- Script Finished ---")
wandb.finish()

### Visualize WandB Evaluation WER Trends
This block fetches the logged metrics across all the different augmentation experiments from Weights & Biases and plots them together for comparison.

In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import userdata

print("Fetching data from WandB...")
try:
    # Retrieve the API key from Colab Secrets and strip any hidden newlines/spaces
    wandb_key = userdata.get('WANDB_API_KEY').strip()

    # Explicitly call login with the key
    wandb.login(key=wandb_key)

    # Initialize the WandB Public API
    api = wandb.Api()

    # Try default entity first, then fallback to the team/org name from the logs
    project = "swahili-asr-whisper-lora"
    entity = api.default_entity

    print(f"Attempting to fetch runs from: {entity}/{project}")
    try:
        runs = api.runs(f"{entity}/{project}")
    except Exception:
        entity = "sairascyrus-laikipia-university"
        print(f"Fallback: Attempting to fetch runs from: {entity}/{project}")
        runs = api.runs(f"{entity}/{project}")

    plt.figure(figsize=(14, 8))
    sns.set_theme(style="whitegrid")

    has_data = False

    # Target run names to include in the plot
    target_runs = [
        "whisper-small-lora-baseline-no-aug",
        "whisper-small-lora-aug-gaussian-only",
        "whisper-small-lora-aug-timestretch-only",
        "whisper-small-lora-aug-pitchshift-only",
        "whisper-small-lora-aug-specaugment-only",
        "whisper-small-lora-aug-all-combined"
    ]

    for run in runs:
        # Skip runs that are not in our target list
        if run.name not in target_runs:
            continue

        # Fetch the step and eval_wer history for each run
        history = run.history(keys=["step", "eval_wer"])
        if not history.empty and "eval_wer" in history.columns and "step" in history.columns:
            history = history.dropna(subset=["step", "eval_wer"])
            if not history.empty:
                has_data = True
                # Plot the trend for this run
                plt.plot(history["step"], history["eval_wer"], label=run.name, marker="o", linewidth=2)

    if has_data:
        # Add a horizontal line for the previous best baseline
        plt.axhline(y=13.32, color='red', linestyle='--', linewidth=2.5, label='Previous Best Baseline (WER: 13.32)')

        plt.title("Evaluation WER Trends across Different Augmentations", fontsize=16)
        plt.xlabel("Training Step", fontsize=14)
        plt.ylabel("Evaluation WER", fontsize=14)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print("No 'eval_wer' data found in the selected runs for this project.")

except userdata.SecretNotFoundError:
    print("WANDB_API_KEY not found in Colab Secrets.")
    print("Please add your WandB API key to the Colab Secrets (the 🔑 icon on the left) with the name 'WANDB_API_KEY'.")
except Exception as e:
    print(f"Failed to fetch data or plot trends: {e}")
    print("Please verify your WandB authentication and project/entity names.")


In [ ]:
import wandb
import pandas as pd
from google.colab import userdata

try:
    wandb_key = userdata.get('WANDB_API_KEY').strip()
    wandb.login(key=wandb_key)
    api = wandb.Api()

    project = "swahili-asr-whisper-lora"
    entity = "sairascyrus-laikipia-university" # Using the fallback entity that worked in the previous cell
    runs = api.runs(f"{entity}/{project}")

    target_runs = [
        "whisper-small-lora-baseline-no-aug",
        "whisper-small-lora-aug-gaussian-only",
        "whisper-small-lora-aug-timestretch-only",
        "whisper-small-lora-aug-pitchshift-only",
        "whisper-small-lora-aug-specaugment-only",
        "whisper-small-lora-aug-all-combined"
    ]

    results = []
    for run in runs:
        if run.name in target_runs:
            history = run.history(keys=["step", "eval_wer"])
            if not history.empty and "eval_wer" in history.columns:
                min_wer = history["eval_wer"].min()
                results.append({"Run Name": run.name, "Best Eval WER": min_wer})

    if results:
        df_results = pd.DataFrame(results).sort_values(by="Best Eval WER")
        display(df_results)
    else:
        print("No eval_wer data found for the target runs.")

except Exception as e:
    print(f"An error occurred: {e}")